# Figure 10

With this notebook we reproduce Figure 10 in Ronchi et al. (2026).
Note that in order to reproduce the plot you first need to simulate a full population considering the best parameters obtained in the corner plot of Figure 6 in Ronchi et al. (2026). To do this you first need to change the following parameters in the `mlpoppyns/simulator/config_simulator.py` file:
```
"NS_number": 3500000,
"t_age_max": 100000000.0,
"P_initial_log10_mean": -0.26,
"P_initial_log10_sigma": 0.7,
"magnetic_field_model": "double_log-normal",
"B_initial_log10_mean_comp1": 12.72,
"B_initial_log10_mean_comp2": 14.11,
"B_initial_log10_sigma_comp1": 0.45,
"B_initial_log10_sigma_comp2": 0.49,
"B_initial_log10_weight_comp1": 0.74,
"a_late": -0.85,
"L_radio_log10_mean": 25.7,
"L_radio_log10_sigma": 0.8,
```
and then launch the simulation with the command:
```
python mlpoppyns/simulator/simulate_population_full.py --save_dir data/paper_results/ronchi_etal_2026/simulation_best_params_full
```

The simulation output will be saved in the path: `data/paper_results/ronchi_etal_2026/simulation_best_params_full`.

In [ ]:
import numpy as np
import pandas as pd
import pathlib
import matplotlib.pyplot as plt
from matplotlib import rcParams
from matplotlib import rc
import matplotlib as mpl
from scipy.integrate import quad
from scipy.optimize import curve_fit
from mlpoppyns.simulator.config_simulator import cfg
import mlpoppyns.simulator.basics.constants as const
import mlpoppyns.simulator.multiband_emission.emission_radio as er

import utilities.plot_settings

In [ ]:
path_to_simulation = pathlib.Path("../../data/paper_results/ronchi_etal_2026/simulation_best_params_full/")

data_full = pd.read_pickle(
    pathlib.Path().joinpath(path_to_simulation, "final_population.pkl.gz"),
    compression="gzip",
)

data_full.columns

In [ ]:
B = data_full["B"]["[G]"].to_numpy()
B_initial = data_full["B_initial"]["[G]"].to_numpy()
P = data_full["P"]["[s]"].to_numpy()

In [ ]:
P_bins = np.logspace(-2.0, 4, 31)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))

ax.hist(
    P[B>=1e14],
    bins=P_bins,
    histtype="step",
    edgecolor="gold",
    lw=4,
    label=r"$B>10^{14}$ G",
)

ax.hist(
    P[(B>=1e13) & (B<1e14)],
    bins=P_bins,
    histtype="step",
    edgecolor="tab:orange",
    lw=4,
    label=r"$10^{13}<B<10^{14}$ G",
)

ax.hist(
    P[(B>=1e12) & (B<1e13)],
    bins=P_bins,
    histtype="step",
    edgecolor="tab:red",
    lw=4,
    label=r"$10^{12}<B<10^{13}$ G",
)

ax.hist(
    P[(B>=1e11) & (B<1e12)],
    bins=P_bins,
    histtype="step",
    edgecolor="tab:brown",
    lw=4,
    label=r"$10^{11}<B<10^{12}$ G",
)
ax.hist(
    P[B<1e11],
    bins=P_bins,
    histtype="step",
    edgecolor="tab:purple",
    lw=4,
    label=r"$B<10^{11}$ G",
)

plt.xlabel(r"$P$ [s]")
plt.ylabel(r"Number of NSs")
# plt.xlim(0., 50.0)
#plt.ylim(0.1, 2.0e5)
plt.xscale("log")
plt.yscale("log")
plt.legend(frameon=False, loc=0, fontsize=20)

plt.savefig("plots/LPT.pdf", dpi=400, bbox_inches="tight")
plt.show()

In [ ]:
print(f"Fraction of NSs with P > 20 s: ", len(P[P>20])/len(P))
print(f"Fraction of NSs with P > 20 s and B > 10^14 G: ", len(P[(P>20) & (B>1e14)])/len(P))
print(f"Fraction of NSs with P > 100 s: ", len(P[P>100])/len(P))